# EXP-02: BERTimbau Large

**Competition:** spr-2026-mammography-report-classification

**Model:** neuralmind/bert-large-portuguese-cased (BERTimbau Large)

**Setup:** Add dataset `dennisfong/neuralmindbert-large-portuguese-cased` to the notebook inputs.

BI-RADS classification (7 classes, 0-6) with GroupKFold CV, macro F1 metric.

In [ ]:
import os, gc, glob, time, hashlib, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

warnings.filterwarnings('ignore')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# === Config ===
import os, glob, shutil

def find_input(pattern):
    """Busca recursivamente em /kaggle/input por pasta que match o pattern."""
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if pattern in d.lower():
                return os.path.join(root, d)
    hits = glob.glob(f'/kaggle/input/**/*{pattern}*', recursive=True)
    return hits[0] if hits else None

MODEL_PATH = find_input('bert-large-portuguese-cased')
COMP_DIR   = find_input('spr-2026-mammography')

assert MODEL_PATH, f'Dataset do modelo nao encontrado! Conteudo: {os.listdir("/kaggle/input")}'
assert COMP_DIR,   f'Competicao nao encontrada! Conteudo: {os.listdir("/kaggle/input")}'

NUM_LABELS = 7
MAX_LEN    = 256
BATCH_SIZE = 8
GRAD_ACCUM = 4
EPOCHS     = 3
LR         = 1e-5
WARMUP     = 0.1
WD         = 0.01
N_SPLITS   = 3

print(f'MODEL_PATH: {MODEL_PATH}')
print(f'COMP_DIR:   {COMP_DIR}')
print(f'Model files: {os.listdir(MODEL_PATH)[:10]}')
print(f'Comp files:  {os.listdir(COMP_DIR)}')

In [ ]:
train_df = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))
print(f'Train: {train_df.shape} | Test: {test_df.shape}')
print(train_df['target'].value_counts().sort_index())

In [ ]:
def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode('utf-8')).hexdigest()

def make_groups(df):
    return df['report'].apply(stable_hash).values

def evaluate(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

class ReportDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len, labels=None):
        self.enc = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt',
        )
        self.labels = labels

    def __len__(self):
        return len(self.enc['input_ids'])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# === GroupKFold Cross-Validation ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
groups = make_groups(train_df)
gkf = GroupKFold(n_splits=N_SPLITS)

oof_preds = np.zeros(len(train_df), dtype=int)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(train_df, train_df['target'], groups)):
    print(f'\n{"="*40} FOLD {fold} {"="*40}')
    tr_texts = train_df.iloc[tr_idx]['report'].values
    va_texts = train_df.iloc[va_idx]['report'].values
    tr_labels = train_df.iloc[tr_idx]['target'].values
    va_labels = train_df.iloc[va_idx]['target'].values

    tr_ds = ReportDataset(tr_texts, tokenizer, MAX_LEN, tr_labels)
    va_ds = ReportDataset(va_texts, tokenizer, MAX_LEN, va_labels)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=NUM_LABELS)

    fold_dir = f'/kaggle/working/fold{fold}'
    args = TrainingArguments(
        output_dir=fold_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_ratio=WARMUP,
        weight_decay=WD,
        fp16=True,
        eval_strategy='epoch',
        save_strategy='no',
        report_to='none',
        logging_steps=50,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {'f1': f1_score(labels, preds, average='macro')}

    trainer = Trainer(
        model=model, args=args,
        train_dataset=tr_ds, eval_dataset=va_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    preds = trainer.predict(va_ds)
    fold_preds = np.argmax(preds.predictions, axis=-1)
    oof_preds[va_idx] = fold_preds
    fold_f1 = evaluate(va_labels, fold_preds)
    fold_scores.append(fold_f1)
    print(f'Fold {fold} F1: {fold_f1:.4f}')

    del model, trainer, tr_ds, va_ds
    gc.collect()
    torch.cuda.empty_cache()
    # Limpa disco — remove checkpoints do fold
    if os.path.exists(fold_dir):
        shutil.rmtree(fold_dir)

print(f'\nOOF macro-F1: {evaluate(train_df["target"].values, oof_preds):.4f}')
print(f'Mean fold F1: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')

In [ ]:
# === Retrain on full data + Generate submission ===
print('Retraining on full data...')
full_ds = ReportDataset(train_df['report'].values, tokenizer, MAX_LEN, train_df['target'].values)
test_ds = ReportDataset(test_df['report'].values, tokenizer, MAX_LEN)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=NUM_LABELS)

args = TrainingArguments(
    output_dir='/kaggle/working/full',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP,
    weight_decay=WD,
    fp16=True,
    save_strategy='no',
    report_to='none',
    logging_steps=50,
)

trainer = Trainer(model=model, args=args, train_dataset=full_ds)
trainer.train()

preds = trainer.predict(test_ds)
test_preds = np.argmax(preds.predictions, axis=-1)

sub = pd.DataFrame({'ID': test_df['ID'], 'target': test_preds})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub['target'].value_counts().sort_index())
print(f'\nSubmission saved: {sub.shape}')
sub.head()